In [ ]:
using Pkg

parentdir = dirname(pwd())
environment_path = string(parentdir)*"/git_GWJulia/GW.jl"
print("Activating environment at ", environment_path)
Pkg.activate(environment_path)
Pkg.instantiate()

using GW

In [ ]:
#Use only if cell below this one returns an error

import Pkg

# List of package names
packages = [
    "Plots",
    "HDF5",
    "Trapz",
    "ForwardDiff",
    "Distributions",
    "LinearAlgebra",
    "Random",
    "ProgressMeter",
    "StatsBase",
    "LaTeXStrings",
    "Healpix",
    "Revise",
    "Interpolations"
]

# Iterate over the list and install each package
for pkg in packages
    Pkg.add(pkg)
end

In [2]:
# using DelimitedFiles
using Plots
# using Interpolations
using HDF5
# using BenchmarkTools
using Trapz
using ForwardDiff
using Distributions
using LinearAlgebra
# using QuadGK
# using Integrals
# using ProgressBars
using Random
# using ProgressMeter
using Base.Threads
# using Polynomials
using StatsBase
using LaTeXStrings
using Plots.PlotMeasures
using Healpix
using Revise


using Interpolations


In [3]:
plot_path_prefix = "plot_overlaid_";

### LVK like events

In [ ]:
#New catalog
nEvents = 100000
typeof_events = "BBH"
catalog_name = "HM_LVK_catalog.h5"

In [ ]:
parameters = GenerateCatalog(nEvents, typeof_events, name_catalog = catalog_name)

In [ ]:
#Read catalog just generated

parameters = ReadCatalog(catalog_name)

parameters = parameters[1:12]


#The last parameter actually is Lambda(1?), therefore I restrict to the first 11, setting to zero the last line (even tought is already was like this...). Then the last entry will be used for the BGR PN deformation parameter
parameters[12] .= 0.0;

In [ ]:
#Loading realistic O3 PSD for LVK

LIGO_L_O3 = Detector(getCoords(LIGO_L)..., 'L', _readASD(environment_path * "/useful_files/LVC_O1O2O3/O3-L1-C01_CLEAN_SUB60HZ-1240573680.0_sensitivity_strain_asd.txt")...,  "LIGO_L_O3")

LIGO_H_O3 = Detector(getCoords(LIGO_H)..., 'L', _readASD(environment_path * "/useful_files/LVC_O1O2O3/O3-H1-C01_CLEAN_SUB60HZ-1251752040.0_sensitivity_strain_asd.txt")...,  "LIGO_H_O3")

VIRGO_O3 = Detector(getCoords(VIRGO)..., 'L', _readASD(environment_path * "/useful_files/LVC_O1O2O3/O3-V1_sensitivity_strain_asd.txt")...,  "VIRGO_O3")

In [ ]:
#Create LV(K) network

println( _available_detectors());
LVKnetwork = [_available_detectors("LIGO_L"), _available_detectors("LIGO_H"), _available_detectors("VIRGO")];

LVKnetwork_O3 = [LIGO_L_O3, LIGO_H_O3, VIRGO_O3];
# LVKnetwork=LVKnetwork_O3

In [ ]:
#Choose available waveform model   

println(_available_waveforms())


wfPhenomD = PhenomD();
println(wfPhenomD)

wfPhenomDTiger = PhenomD_TIGER(0.);
println(wfPhenomDTiger)

In [ ]:
#BGR PN orders
source = "BBH_LVK"
nEvents = "100k_CUT"
folder = "BGR_LVK/"

PNorder_ = [-1, 0, 0.5, 1, 1.5, 2, log(2.5), 3, log(3.), 3.5]
name_folder = folder .*["minus_one", "zero", "half", "one", "one_half", "two", "log_two_half", "three", "log_three", "three_half"] .* "/"
name_network = ["LVK"]
name_PN = ["-1", "0", "0.5", "1", "1.5", "2", "log(2.5)", "3", "log(3.)", "3.5"];

name_folder

In [ ]:
snrs = SNR(wfPhenomD, LVKnetwork, parameters...);

In [ ]:
# I select only the event in the catalog (so the 12 ntuple of vectors in parameters) that have SNR > 12 in all the detectors, comparing the count of events before and after
# Actually not necessary, the Fisher matrix code below already does this! Yet at least in this way you may be more consistent?

SNR_cut = 12
parametersCut = ntuple(i -> parameters[i][snrs .> SNR_cut], length(parameters))

println("Number of events before selection: ", length(parameters[1]))
println("Number of events after selection: ", length(parametersCut[1]))


In [ ]:
#Evaluates and saves Fisher and SNR for the events in the catalog which pass the SNR cut (should be applied also to the inspiral actually)

println("folder = ", folder);
println(name_folder)

for (ii, PNorder) in enumerate(PNorder_)
    name = name_folder[ii]
    println("PNorder = ", PNorder)
    for (jj, network) in enumerate(name_network)
        name_ = name * name_network[jj] 
        println("network = ", name_network[jj])
        println("name_ = ", name_)
        #if name_ == "BGR/minus_one/network_0_15km" || name_ == "BGR/minus_one/network_45_15km" 
        # if PNorder == -1 
        #     println("Skipping")
        #     global jj+=1
        #     continue
        # end

        # rho_thres=12
        @time FisherMatrix(PhenomD_TIGER(PNorder), LVKnetwork, parametersCut..., auto_save=true, return_SNR=true, name_folder=name_, useEarthMotion=true)
        
    end
end


In [ ]:
#Reads the previously evaluates SNRs, Fisher matrices, and computes other relevant quantities, such as the errors

Fisher = zeros(length(PNorder_), length(name_network), length(parametersCut[1]), length(parametersCut), length(parametersCut))
cov = zeros(length(PNorder_), length(name_network), length(parametersCut[1]), length(parametersCut), length(parametersCut))
SNRs = zeros(length(PNorder_), length(name_network), length(parametersCut[1]))
errors = zeros(length(PNorder_), length(name_network), length(parametersCut[1]), length(parametersCut))

has_covariance_matrix_already_been_computed = false

for (ii, PNorder) in enumerate(PNorder_)
    name = name_folder[ii]
    println("PNorder = ", PNorder)
    for (jj, network) in enumerate(name_network)
        name_ = name * network
        println("network = ", network)
        println("name_ = ", name_);

        Fisher[ii,jj,:,:,:], SNRs[ii,jj,:] = _read_Fishers_SNRs("output/"*name_*"/Fishers_SNRs.h5")

        if has_covariance_matrix_already_been_computed == false
            # Calculate the covariance matrix
            cov[ii,jj,:,:,:] = CovMatrix(Fisher[ii,jj,:,:,:])
            
            # save the covariance matrix
            h5open("output/"*name_*"/CovMatrix.h5", "w") do file
                write(file, "cov", cov[ii,jj,:,:,:])
            end
        end

        # read covariance matrix
        h5open("output/"*name_*"/CovMatrix.h5", "r") do file
            cov[ii,jj,:,:,:] = read(file, "cov")
        end

        errors[ii,jj,:,:] = Errors(cov[ii,jj,:,:,:])
        println("jj = ", jj)

    end
end

In [ ]:
#Function to remove an event altogether (from all PN orders and detector configurations) if that event return a zero error for any PN order on any detector networks (I completely discard any event that could be problematic)!

function remove_zero_entries!(PN_errors)
    # Get the size of the array
    size_ii, size_jj, size_kk = size(PN_errors)
    
    # Create a list to store indices of kk to be removed
    indices_to_keep = trues(size_kk)
    
    # Iterate over the kk dimension
    for kk in 1:size_kk
        # Check if any element in the ii or jj dimensions is zero
        for ii in 1:size_ii
            for jj in 1:size_jj
                if PN_errors[ii, jj, kk] == 0
                    indices_to_keep[kk] = false
                    break
                end
            end
            if !indices_to_keep[kk]
                break
            end
        end 
    end
    
    println("Removing ", count(!, indices_to_keep), " events altogether!" )

    # Create a new array without the identified kk entries
    PN_errors = PN_errors[:, :, indices_to_keep]

    println(length(PN_errors[1,1,:]), " elements remaining!")
    
    return PN_errors
end

In [ ]:
# cumulative error on PNorder
cumulative_error = zeros(length(PNorder_), length(name_network))

#Now I restrict the errors just to the PN deformation parameter!
PN_errors = errors[:, :, :, end]

#And I discard altogether any event that may be problematic (has zero error for any PN order on any detector configuration)
PN_errors = remove_zero_entries!(PN_errors)

println(size(PN_errors))

for (ii, PNorder) in enumerate(PNorder_)
    println("PNorder = ", PNorder)
    for (jj, network) in enumerate(name_network)
        println("network = ", network)

        vecc = PN_errors[ii,jj,:];

        # remove zeros... actually there should be no zeros!?!    
        zero_count = count(x -> x == 0, vecc)
        if zero_count > 0
        println("ERROR: The array STILL contains $zero_count zero elements.")
            vecc = vecc[vecc .!= 0]
        end

        cumulative_error[ii,jj] = sum(vecc.^(-2))^-0.5
    end
end

In [ ]:
using Plots
using LaTeXStrings  # Proper LaTeX font support

# Use LaTeX-like fonts
default(fontfamily="Computer Modern", titlefontsize=12, guidefontsize=10, tickfontsize=8, legendfontsize=9)

# Proper LaTeX labels
xlabel_str = L"\mathrm{PN~Order}"  # Use \mathrm for proper LaTeX rendering
ylabel_str = L"\mathrm{Cumulative~Error}"

# Define a list of different markers to enhance readability
markers = [:circle, :square, :diamond, :utriangle, :dtriangle, :hexagon]

# Initialize the plot with white background and high resolution
cc = plot(
    xlabel=xlabel_str, ylabel=ylabel_str, title="Cumulative Error on PN Order",
    legend=:bottomright, xticks=(1:10, name_PN), 
    yscale=:log10, size=(900, 600), dpi=300,
    grid=true, framestyle=:box
)

# Convert PN order to indices for plotting
PNorder_plot = collect(1:length(PNorder_))

# Loop through each network and scatter with unique markers
for jj in 1:length(name_network)
    scatter!(
        cc, PNorder_plot, cumulative_error[:, jj],
        label=name_network[jj],
        marker=markers[mod1(jj, length(markers))],  # Cycle through marker list
        markersize=6, markerstrokewidth=1, markerstrokecolor=:black,
        alpha=0.9  # Slight transparency for overlapping points
    )
end

# Display the final plot
display(cc)

#### Reproduce the code for LVK with just a fixed number of events, but repeatedly, to produce also error bars

In [ ]:
# I recycle the errors already computed, just cutting the arrays into subarrays!
NUMBER_OF_EVENTS_TO_BE_USED_SINGLE_CATALOG = 12

num_groups = div(length(PN_errors[1,1,:]), NUMBER_OF_EVENTS_TO_BE_USED_SINGLE_CATALOG)

println("Number of groups/subdivision that will allow to evaluate the errorbars on the errors: $num_groups")

cumulative_split_error = zeros(num_groups, length(PNorder_), length(name_network))

for (ii, PNorder) in enumerate(PNorder_)
    println("PNorder = ", PNorder)
    for (jj, network) in enumerate(name_network)
        println("network = ", network)

        vecc = PN_errors[ii,jj,:];

        # remove zeros... actually there should be no zeros!?!    
        zero_count = count(x -> x == 0, vecc)
        if zero_count > 0
        println("ERROR: The array STILL contains $zero_count zero elements.")
            vecc = vecc[vecc .!= 0]
        end

        # Ensure vecc has enough elements after removing zeros
        if length(vecc) < num_groups * NUMBER_OF_EVENTS_TO_BE_USED_SINGLE_CATALOG
            println("ERROR: Not enough elements in vecc after removing zeros.")
            continue
        end

        for nn in 1:num_groups
            start_idx = (nn - 1) * NUMBER_OF_EVENTS_TO_BE_USED_SINGLE_CATALOG + 1
            end_idx = nn * NUMBER_OF_EVENTS_TO_BE_USED_SINGLE_CATALOG
            cumulative_split_error[nn, ii, jj] = sum((vecc[start_idx:end_idx]).^(-2))^-0.5
        end

    end
end

In [ ]:
import Statistics: mean, std  # Explicitly import only the required functions
using Plots
using LaTeXStrings  # Proper LaTeX font support

# Use LaTeX-like fonts
default(fontfamily="Computer Modern", titlefontsize=12, guidefontsize=10, tickfontsize=8, legendfontsize=9)

# Proper LaTeX labels
xlabel_str = L"\mathrm{PN~Order}"  # Use \mathrm for proper LaTeX rendering
ylabel_str = L"\mathrm{Cumulative~Error}"

# Define a list of different markers to enhance readability
markers = [:circle, :square, :diamond, :utriangle, :dtriangle, :hexagon]

# Initialize the plot with white background and high resolution
cc = plot(
    xlabel=xlabel_str, ylabel=ylabel_str, title="Cumulative Error on PN Order",
    legend=:bottomright, xticks=(1:10, name_PN), 
    yscale=:log10, size=(900, 600), dpi=300,
    grid=true, framestyle=:box
)

# Convert PN order to indices for plotting
PNorder_plot = collect(1:length(PNorder_))

# Calculate mean and standard deviation for each ii and jj
mean_cumulative_error = zeros(length(PNorder_), length(name_network))
std_cumulative_error = zeros(length(PNorder_), length(name_network))

for ii in 1:length(PNorder_)
    for jj in 1:length(name_network)
        mean_cumulative_error[ii, jj] = mean(cumulative_split_error[:, ii, jj])
        std_cumulative_error[ii, jj] = std(cumulative_split_error[:, ii, jj])
    end
end

# Loop through each network and scatter with unique markers
for jj in 1:length(name_network)
    scatter!(
        cc, PNorder_plot, mean_cumulative_error[:, jj],
        yerr=std_cumulative_error[:, jj],  # Add error bars
        label=name_network[jj],
        marker=markers[mod1(jj, length(markers))],  # Cycle through marker list
        markersize=6, markerstrokewidth=1, markerstrokecolor=:black,
        alpha=0.9  # Slight transparency for overlapping points
    )
end

# Save the plot to a file on the remote server
savefig(cc, plot_path_prefix * "cumulative_PN_error_plot_PhenomD.png") 

# Display the final plot
display(cc)


In [ ]:
import Statistics: mean, std  # Explicitly import only the required functions
using Plots
using LaTeXStrings  # Proper LaTeX font support

# Use LaTeX-like fonts
default(fontfamily="Computer Modern", titlefontsize=12, guidefontsize=10, tickfontsize=8, legendfontsize=9)

# Proper LaTeX labels
xlabel_str = L"\mathrm{PN~Order}"  # Use \mathrm for proper LaTeX rendering
ylabel_str = L"\mathrm{Cumulative~Error}"

# Define a list of different markers to enhance readability
markers = [:circle, :square, :diamond, :utriangle, :dtriangle, :hexagon]

# Calculate mean and standard deviation for each ii and jj
mean_cumulative_error = zeros(length(PNorder_), length(name_network))
std_cumulative_error = zeros(length(PNorder_), length(name_network))

for ii in 1:length(PNorder_)
    for jj in 1:length(name_network)
        mean_cumulative_error[ii, jj] = mean(cumulative_split_error[:, ii, jj])
        std_cumulative_error[ii, jj] = std(cumulative_split_error[:, ii, jj])
    end
end

# Find the minimum and maximum values for y-axis limits
min_y = minimum(mean_cumulative_error) / 10  # Divide by 10 for some padding
max_y = maximum(mean_cumulative_error) * 10  # Multiply by 10 for some padding

# Initialize the plot with white background and high resolution
cc = plot(
    xlabel=xlabel_str, ylabel=ylabel_str, title="Cumulative Error on PN Order - PhenomD LVK default",
    legend=:bottomright, xticks=(1:10, name_PN), 
    yscale=:log10, size=(900, 600), dpi=300,
    grid=true, framestyle=:box,
    yticks=[10.0^i for i in floor(Int, log10(min_y)):ceil(Int, log10(max_y))],  # Set y-axis ticks for each 10^N value within the range
    ylims=(min_y, max_y)  # Automatically set the y-axis range
)

# Convert PN order to indices for plotting
PNorder_plot = collect(1:length(PNorder_))

# Loop through each network and scatter with unique markers
for jj in 1:length(name_network)
    scatter!(
        cc, PNorder_plot, mean_cumulative_error[:, jj],
        yerr=std_cumulative_error[:, jj],  # Add error bars
        label=name_network[jj],
        marker=markers[mod1(jj, length(markers))],  # Cycle through marker list
        markersize=6, markerstrokewidth=1, markerstrokecolor=:black,
        alpha=0.9  # Slight transparency for overlapping points
    )
    
    # Plot individual errors as small transparent orange horizontal stripes
    for ii in 1:length(PNorder_)
        vecc = PN_errors[ii, jj, 1:NUMBER_OF_EVENTS_TO_BE_USED_SINGLE_CATALOG]
        scatter!(
            cc, fill(PNorder_plot[ii], length(vecc)), vecc,
            marker=:hline, markersize=15, markerstrokewidth=2, alpha=0.5, color=:orange, label=""
        )
    end
end

# Add a dummy plot for the legend entry
scatter!(
    cc, [NaN], [NaN],
    marker=:hline, markersize=15, markerstrokewidth=2, alpha=0.5, color=:orange, label="Errors from single events"
)

# Save the plot to a file on the remote server
savefig(cc, plot_path_prefix * "cumulative_PN_error_plot_PhenomD_single_errors.png")

# Display the final plot
display(cc)

Now with overlaid LVK GWTC-3 results as well

In [ ]:
#Results from figure 6 of https://arxiv.org/pdf/2112.06861

LVK_GWTC3_results = [0.75e-4, 0.06, 0.15, 0.1, 0.07, 0.55, 0.23, 0.48, 2.0, 1.0]

In [ ]:
import Statistics: mean, std  # Explicitly import only the required functions
using Plots
using LaTeXStrings  # Proper LaTeX font support

# Use LaTeX-like fonts
default(fontfamily="Computer Modern", titlefontsize=12, guidefontsize=10, tickfontsize=8, legendfontsize=9)

# Proper LaTeX labels
xlabel_str = L"\mathrm{PN~Order}"  # Use \mathrm for proper LaTeX rendering
ylabel_str = L"\mathrm{Cumulative~Error}"

# Define a list of different markers to enhance readability
markers = [:circle, :square, :diamond, :utriangle, :dtriangle, :hexagon]

# Calculate mean and standard deviation for each ii and jj
mean_cumulative_error = zeros(length(PNorder_), length(name_network))
std_cumulative_error = zeros(length(PNorder_), length(name_network))

for ii in 1:length(PNorder_)
    for jj in 1:length(name_network)
        mean_cumulative_error[ii, jj] = mean(cumulative_split_error[:, ii, jj])
        std_cumulative_error[ii, jj] = std(cumulative_split_error[:, ii, jj])
    end
end

plot_padding_top= 10;
plot_padding_bottom = 3;

# Find the minimum and maximum values for y-axis limits, excluding the first point
min_y = minimum(mean_cumulative_error[2:end, :]) / plot_padding_bottom  # Divide by 10 for some padding
max_y = maximum(mean_cumulative_error[2:end, :]) * plot_padding_top  # Multiply by 10 for some padding

# Find the minimum and maximum values for y-axis limits, excluding the first point
min_y_sub = minimum([mean_cumulative_error[1], LVK_GWTC3_results[1]]) / plot_padding_bottom  # Divide by 10 for some padding
max_y_sub = maximum([mean_cumulative_error[1], LVK_GWTC3_results[1]]) * plot_padding_top  # Multiply by 10 for some padding

# Initialize the main plot with white background and high resolution
cc_main = plot(
    xlabel=xlabel_str, title="Cumulative Error on PN Order - PhenomD LVK default",
    legend=:bottomright, xticks=(2:10, name_PN[2:end]), 
    yscale=:log10, size=(900, 600), dpi=300,
    xlims=(1.5, 10.5),
    grid=true, framestyle=:box,
    yticks=[10.0^i for i in floor(Int, log10(min_y)):ceil(Int, log10(max_y))],  # Set y-axis ticks for each 10^N value within the range
    ylims=(min_y, max_y),  # Automatically set the y-axis range
    gridalpha=0.5, gridcolor=:gray,  # Set grid lines to be transparent or gray
    yminorgrid=true, minorgridalpha=0.3, minorgridcolor=:gray,  # Enable minor grid lines
    xminorgrid=false  # Disable minor grid lines for the x-axis
)

# Initialize the subplot for the first point
cc_sub = plot(
    xlabel=xlabel_str, ylabel=ylabel_str,
    legend=false, xticks=([1], [name_PN[1]]), 
    xlims=(0.5, 1.5),
    yscale=:log10, size=(300, 600), dpi=300,
    grid=true, framestyle=:box,
    yticks=[10.0^i for i in floor(Int, log10(min_y_sub)) : ceil(Int, log10(max_y_sub))],  # Set y-axis ticks for the first point
    ylims=(min_y_sub, max_y_sub),  # Automatically set the y-axis range
    gridalpha=0.5, gridcolor=:gray,  # Set grid lines to be transparent or gray
    yminorgrid=true, minorgridalpha=0.3, minorgridcolor=:gray,  # Enable minor grid lines
    xminorgrid=false  # Disable minor grid lines for the x-axis
)

# Plot the first point in the subplot
for jj in 1:length(name_network)
    scatter!(
        cc_sub, [1], [mean_cumulative_error[1, jj]],
        yerr=[std_cumulative_error[1, jj]],  # Add error bars
        label=name_network[jj],
        marker=markers[mod1(jj, length(markers))],  # Cycle through marker list
        markersize=6, markerstrokewidth=1, markerstrokecolor=:black,
        alpha=0.9  # Slight transparency for overlapping points
    )
end

# Plot the remaining points in the main plot
for jj in 1:length(name_network)
    scatter!(
        cc_main, 2:length(PNorder_), mean_cumulative_error[2:end, jj],
        yerr=std_cumulative_error[2:end, jj],  # Add error bars
        label=name_network[jj] * " forecast",
        marker=markers[mod1(jj, length(markers))],  # Cycle through marker list
        markersize=6, markerstrokewidth=1, markerstrokecolor=:black,
        alpha=0.9  # Slight transparency for overlapping points
    )
    
    # Plot individual errors as small transparent orange horizontal stripes
    for ii in 1:1
        vecc = PN_errors[ii, jj, 1:NUMBER_OF_EVENTS_TO_BE_USED_SINGLE_CATALOG]
        scatter!(
            cc_sub, fill(ii, length(vecc)), vecc,
            marker=:hline, markersize=15, markerstrokewidth=2, alpha=0.5, color=:orange, label=""
        )
    end

    for ii in 2:length(PNorder_)
        vecc = PN_errors[ii, jj, 1:NUMBER_OF_EVENTS_TO_BE_USED_SINGLE_CATALOG]
        scatter!(
            cc_main, fill(ii, length(vecc)), vecc,
            marker=:hline, markersize=15, markerstrokewidth=2, alpha=0.5, color=:orange, label=""
        )
    end
end

# Add a dummy plot for the legend entry in the main plot
scatter!(
    cc_main, [NaN], [NaN],
    marker=:hline, markersize=15, markerstrokewidth=2, alpha=0.5, color=:orange, label="Errors from single events"
)

# Plot the GWTC-3 results in the main plot
scatter!(
    cc_main, 2:length(PNorder_), LVK_GWTC3_results[2:end],
    label="LVK GWTC-3", marker=:diamond, markersize=6, markerstrokewidth=1, markerstrokecolor=:black, color=:darkblue
)

# Plot the first GWTC-3 result in the subplot
scatter!(
    cc_sub, [1], [LVK_GWTC3_results[1]],
    label="LVK GWTC-3", marker=:diamond, markersize=6, markerstrokewidth=1, markerstrokecolor=:black, color=:darkblue
)

# Combine the main plot and the subplot
plot(cc_sub, cc_main, layout = @layout [a{0.15w} b{0.85w}])

# Save the combined plot to a file on the remote server
savefig(plot_path_prefix * "cumulative_PN_error_plot_PhenomD_single_errors_LVK_GWTC3_overlaid.png")

# Display the final plot
display(plot(cc_sub, cc_main, layout = @layout [a{0.15w} b{0.85w}]))

### Repeat for PhenomHM

In [ ]:
wfPhenomHMTiger = PhenomHM_TIGER(0.);
println(wfPhenomHMTiger)

In [ ]:
#BGR PN orders for HM
source = "BBH_HM_LVK"
nEvents = "100k_CUT"
folder = "BGR_LVK_HM/"

PNorder_ = [-1, 0, 0.5, 1, 1.5, 2, log(2.5), 3, log(3.), 3.5]
name_folder = folder .*["minus_one", "zero", "half", "one", "one_half", "two", "log_two_half", "three", "log_three", "three_half"] .* "/"
name_network = ["LVK"]
name_PN = ["-1", "0", "0.5", "1", "1.5", "2", "log(2.5)", "3", "log(3.)", "3.5"];

name_folder

In [ ]:
#Evaluates and saves Fisher and SNR for the events in the catalog which pass the SNR cut (should be applied also to the inspiral actually)

println("folder = ", folder);
println(name_folder)

for (ii, PNorder) in enumerate(PNorder_)
    name = name_folder[ii]
    println("PNorder = ", PNorder)
    for (jj, network) in enumerate(name_network)
        name_ = name * name_network[jj] 
        println("network = ", name_network[jj])
        println("name_ = ", name_)
        #if name_ == "BGR/minus_one/network_0_15km" || name_ == "BGR/minus_one/network_45_15km" 
        # if PNorder == -1 
        #     println("Skipping")
        #     global jj+=1
        #     continue
        # end

        # rho_thres=12
        @time FisherMatrix(PhenomHM_TIGER(PNorder), LVKnetwork, parametersCut..., auto_save=true, return_SNR=true, name_folder=name_, useEarthMotion=true)
        
    end
end


In [ ]:
#Reads the previously evaluates SNRs, Fisher matrices, and computes other relevant quantities, such as the errors

Fisher = zeros(length(PNorder_), length(name_network), length(parametersCut[1]), length(parametersCut), length(parametersCut))
cov = zeros(length(PNorder_), length(name_network), length(parametersCut[1]), length(parametersCut), length(parametersCut))
SNRs = zeros(length(PNorder_), length(name_network), length(parametersCut[1]))
errors = zeros(length(PNorder_), length(name_network), length(parametersCut[1]), length(parametersCut))

has_covariance_matrix_already_been_computed = false

for (ii, PNorder) in enumerate(PNorder_)
    name = name_folder[ii]
    println("PNorder = ", PNorder)
    for (jj, network) in enumerate(name_network)
        name_ = name * network
        println("network = ", network)
        println("name_ = ", name_);

        Fisher[ii,jj,:,:,:], SNRs[ii,jj,:] = _read_Fishers_SNRs("output/"*name_*"/Fishers_SNRs.h5")

        if has_covariance_matrix_already_been_computed == false
            # Calculate the covariance matrix
            cov[ii,jj,:,:,:] = CovMatrix(Fisher[ii,jj,:,:,:])
            
            # save the covariance matrix
            h5open("output/"*name_*"/CovMatrix.h5", "w") do file
                write(file, "cov", cov[ii,jj,:,:,:])
            end
        end

        # read covariance matrix
        h5open("output/"*name_*"/CovMatrix.h5", "r") do file
            cov[ii,jj,:,:,:] = read(file, "cov")
        end

        errors[ii,jj,:,:] = Errors(cov[ii,jj,:,:,:])
        println("jj = ", jj)

    end
end

In [ ]:
# cumulative error on PNorder
cumulative_error = zeros(length(PNorder_), length(name_network))

#Now I restrict the errors just to the PN deformation parameter!
PN_errors = errors[:, :, :, end]

#And I discard altogether any event that may be problematic (has zero error for any PN order on any detector configuration)
PN_errors = remove_zero_entries!(PN_errors)

println(size(PN_errors))

for (ii, PNorder) in enumerate(PNorder_)
    println("PNorder = ", PNorder)
    for (jj, network) in enumerate(name_network)
        println("network = ", network)

        vecc = PN_errors[ii,jj,:];

        # remove zeros... actually there should be no zeros!?!    
        zero_count = count(x -> x == 0, vecc)
        if zero_count > 0
        println("ERROR: The array STILL contains $zero_count zero elements.")
            vecc = vecc[vecc .!= 0]
        end

        cumulative_error[ii,jj] = sum(vecc.^(-2))^-0.5
    end
end

In [ ]:
using Plots
using LaTeXStrings  # Proper LaTeX font support

# Use LaTeX-like fonts
default(fontfamily="Computer Modern", titlefontsize=12, guidefontsize=10, tickfontsize=8, legendfontsize=9)

# Proper LaTeX labels
xlabel_str = L"\mathrm{PN~Order}"  # Use \mathrm for proper LaTeX rendering
ylabel_str = L"\mathrm{Cumulative~Error}"

# Define a list of different markers to enhance readability
markers = [:circle, :square, :diamond, :utriangle, :dtriangle, :hexagon]

# Initialize the plot with white background and high resolution
cc = plot(
    xlabel=xlabel_str, ylabel=ylabel_str, title="Cumulative Error on PN Order",
    legend=:bottomright, xticks=(1:10, name_PN), 
    yscale=:log10, size=(900, 600), dpi=300,
    grid=true, framestyle=:box
)

# Convert PN order to indices for plotting
PNorder_plot = collect(1:length(PNorder_))

# Loop through each network and scatter with unique markers
for jj in 1:length(name_network)
    scatter!(
        cc, PNorder_plot, cumulative_error[:, jj],
        label=name_network[jj],
        marker=markers[mod1(jj, length(markers))],  # Cycle through marker list
        markersize=6, markerstrokewidth=1, markerstrokecolor=:black,
        alpha=0.9  # Slight transparency for overlapping points
    )
end

# Display the final plot
display(cc)

In [ ]:
# I recycle the errors already computed, just cutting the arrays into subarrays!
NUMBER_OF_EVENTS_TO_BE_USED_SINGLE_CATALOG = 12

num_groups = div(length(PN_errors[1,1,:]), NUMBER_OF_EVENTS_TO_BE_USED_SINGLE_CATALOG)

println("Number of groups/subdivision that will allow to evaluate the errorbars on the errors: $num_groups")

cumulative_split_error = zeros(num_groups, length(PNorder_), length(name_network))

for (ii, PNorder) in enumerate(PNorder_)
    println("PNorder = ", PNorder)
    for (jj, network) in enumerate(name_network)
        println("network = ", network)

        vecc = PN_errors[ii,jj,:];

        # remove zeros... actually there should be no zeros!?!    
        zero_count = count(x -> x == 0, vecc)
        if zero_count > 0
        println("ERROR: The array STILL contains $zero_count zero elements.")
            vecc = vecc[vecc .!= 0]
        end

        # Ensure vecc has enough elements after removing zeros
        if length(vecc) < num_groups * NUMBER_OF_EVENTS_TO_BE_USED_SINGLE_CATALOG
            println("ERROR: Not enough elements in vecc after removing zeros.")
            continue
        end

        for nn in 1:num_groups
            start_idx = (nn - 1) * NUMBER_OF_EVENTS_TO_BE_USED_SINGLE_CATALOG + 1
            end_idx = nn * NUMBER_OF_EVENTS_TO_BE_USED_SINGLE_CATALOG
            cumulative_split_error[nn, ii, jj] = sum((vecc[start_idx:end_idx]).^(-2))^-0.5
        end

    end
end

In [ ]:
import Statistics: mean, std  # Explicitly import only the required functions
using Plots
using LaTeXStrings  # Proper LaTeX font support

# Use LaTeX-like fonts
default(fontfamily="Computer Modern", titlefontsize=12, guidefontsize=10, tickfontsize=8, legendfontsize=9)

# Proper LaTeX labels
xlabel_str = L"\mathrm{PN~Order}"  # Use \mathrm for proper LaTeX rendering
ylabel_str = L"\mathrm{Cumulative~Error}"

# Define a list of different markers to enhance readability
markers = [:circle, :square, :diamond, :utriangle, :dtriangle, :hexagon]

# Initialize the plot with white background and high resolution
cc = plot(
    xlabel=xlabel_str, ylabel=ylabel_str, title="Cumulative Error on PN Order",
    legend=:bottomright, xticks=(1:10, name_PN), 
    yscale=:log10, size=(900, 600), dpi=300,
    grid=true, framestyle=:box
)

# Convert PN order to indices for plotting
PNorder_plot = collect(1:length(PNorder_))

# Calculate mean and standard deviation for each ii and jj
mean_cumulative_error = zeros(length(PNorder_), length(name_network))
std_cumulative_error = zeros(length(PNorder_), length(name_network))

for ii in 1:length(PNorder_)
    for jj in 1:length(name_network)
        mean_cumulative_error[ii, jj] = mean(cumulative_split_error[:, ii, jj])
        std_cumulative_error[ii, jj] = std(cumulative_split_error[:, ii, jj])
    end
end

# Loop through each network and scatter with unique markers
for jj in 1:length(name_network)
    scatter!(
        cc, PNorder_plot, mean_cumulative_error[:, jj],
        yerr=std_cumulative_error[:, jj],  # Add error bars
        label=name_network[jj],
        marker=markers[mod1(jj, length(markers))],  # Cycle through marker list
        markersize=6, markerstrokewidth=1, markerstrokecolor=:black,
        alpha=0.9  # Slight transparency for overlapping points
    )
end

# Save the plot to a file on the remote server
savefig(cc, plot_path_prefix *  "cumulative_PN_error_plot_PhenomHM.png")

# Display the final plot
display(cc)


In [ ]:
import Statistics: mean, std  # Explicitly import only the required functions
using Plots
using LaTeXStrings  # Proper LaTeX font support

# Use LaTeX-like fonts
default(fontfamily="Computer Modern", titlefontsize=12, guidefontsize=10, tickfontsize=8, legendfontsize=9)

# Proper LaTeX labels
xlabel_str = L"\mathrm{PN~Order}"  # Use \mathrm for proper LaTeX rendering
ylabel_str = L"\mathrm{Cumulative~Error}"

# Define a list of different markers to enhance readability
markers = [:circle, :square, :diamond, :utriangle, :dtriangle, :hexagon]

# Calculate mean and standard deviation for each ii and jj
mean_cumulative_error = zeros(length(PNorder_), length(name_network))
std_cumulative_error = zeros(length(PNorder_), length(name_network))

for ii in 1:length(PNorder_)
    for jj in 1:length(name_network)
        mean_cumulative_error[ii, jj] = mean(cumulative_split_error[:, ii, jj])
        std_cumulative_error[ii, jj] = std(cumulative_split_error[:, ii, jj])
    end
end

# Find the minimum and maximum values for y-axis limits
min_y = minimum(mean_cumulative_error) / 10  # Divide by 10 for some padding
max_y = maximum(mean_cumulative_error) * 10  # Multiply by 10 for some padding

# Initialize the plot with white background and high resolution
cc = plot(
    xlabel=xlabel_str, ylabel=ylabel_str, title="Cumulative Error on PN Order - PhenomHM LVK default",
    legend=:bottomright, xticks=(1:10, name_PN), 
    yscale=:log10, size=(900, 600), dpi=300,
    grid=true, framestyle=:box,
    yticks=[10.0^i for i in floor(Int, log10(min_y)):ceil(Int, log10(max_y))],  # Set y-axis ticks for each 10^N value within the range
    ylims=(min_y, max_y)  # Automatically set the y-axis range
)

# Convert PN order to indices for plotting
PNorder_plot = collect(1:length(PNorder_))

# Loop through each network and scatter with unique markers
for jj in 1:length(name_network)
    scatter!(
        cc, PNorder_plot, mean_cumulative_error[:, jj],
        yerr=std_cumulative_error[:, jj],  # Add error bars
        label=name_network[jj],
        marker=markers[mod1(jj, length(markers))],  # Cycle through marker list
        markersize=6, markerstrokewidth=1, markerstrokecolor=:black,
        alpha=0.9  # Slight transparency for overlapping points
    )
    
    # Plot individual errors as small transparent orange horizontal stripes
    for ii in 1:length(PNorder_)
        vecc = PN_errors[ii, jj, 1:NUMBER_OF_EVENTS_TO_BE_USED_SINGLE_CATALOG]
        scatter!(
            cc, fill(PNorder_plot[ii], length(vecc)), vecc,
            marker=:hline, markersize=15, markerstrokewidth=2, alpha=0.5, color=:orange, label=""
        )
    end
end

# Add a dummy plot for the legend entry
scatter!(
    cc, [NaN], [NaN],
    marker=:hline, markersize=15, markerstrokewidth=2, alpha=0.5, color=:orange, label="Errors from single events"
)

# Save the plot to a file on the remote server
savefig(cc, plot_path_prefix *  "cumulative_PN_error_plot_PhenomHM_single_errors.png")

# Display the final plot
display(cc)

In [ ]:
import Statistics: mean, std  # Explicitly import only the required functions
using Plots
using LaTeXStrings  # Proper LaTeX font support

# Use LaTeX-like fonts
default(fontfamily="Computer Modern", titlefontsize=12, guidefontsize=10, tickfontsize=8, legendfontsize=9)

# Proper LaTeX labels
xlabel_str = L"\mathrm{PN~Order}"  # Use \mathrm for proper LaTeX rendering
ylabel_str = L"\mathrm{Cumulative~Error}"

# Define a list of different markers to enhance readability
markers = [:circle, :square, :diamond, :utriangle, :dtriangle, :hexagon]

# Calculate mean and standard deviation for each ii and jj
mean_cumulative_error = zeros(length(PNorder_), length(name_network))
std_cumulative_error = zeros(length(PNorder_), length(name_network))

for ii in 1:length(PNorder_)
    for jj in 1:length(name_network)
        mean_cumulative_error[ii, jj] = mean(cumulative_split_error[:, ii, jj])
        std_cumulative_error[ii, jj] = std(cumulative_split_error[:, ii, jj])
    end
end

plot_padding_top= 10;
plot_padding_bottom = 3;

# Find the minimum and maximum values for y-axis limits, excluding the first point
min_y = minimum(mean_cumulative_error[2:end, :]) / plot_padding_bottom  # Divide by 10 for some padding
max_y = maximum(mean_cumulative_error[2:end, :]) * plot_padding_top  # Multiply by 10 for some padding


# Find the minimum and maximum values for y-axis limits, excluding the first point
min_y_sub = minimum([mean_cumulative_error[1], LVK_GWTC3_results[1]]) / plot_padding_bottom  # Divide by 10 for some padding
max_y_sub = maximum([mean_cumulative_error[1], LVK_GWTC3_results[1]]) * plot_padding_top  # Multiply by 10 for some padding

# Initialize the main plot with white background and high resolution
cc_main = plot(
    xlabel=xlabel_str, title="Cumulative Error on PN Order - PhenomHM LVK default",
    legend=:bottomright, xticks=(2:10, name_PN[2:end]), 
    yscale=:log10, size=(900, 600), dpi=300,
    xlims=(1.5, 10.5),
    grid=true, framestyle=:box,
    yticks=[10.0^i for i in floor(Int, log10(min_y)):ceil(Int, log10(max_y))],  # Set y-axis ticks for each 10^N value within the range
    ylims=(min_y, max_y),  # Automatically set the y-axis range
    gridalpha=0.5, gridcolor=:gray,  # Set grid lines to be transparent or gray
    yminorgrid=true, minorgridalpha=0.3, minorgridcolor=:gray,  # Enable minor grid lines
    xminorgrid=false  # Disable minor grid lines for the x-axis
)

# Initialize the subplot for the first point
cc_sub = plot(
    xlabel=xlabel_str, ylabel=ylabel_str,
    legend=false, xticks=([1], [name_PN[1]]), 
    xlims=(0.5, 1.5),
    yscale=:log10, size=(300, 600), dpi=300,
    grid=true, framestyle=:box,
    yticks=[10.0^i for i in floor(Int, log10(min_y_sub)) : ceil(Int, log10(max_y_sub))],  # Set y-axis ticks for the first point
    ylims=(min_y_sub, max_y_sub),  # Automatically set the y-axis range
    gridalpha=0.5, gridcolor=:gray,  # Set grid lines to be transparent or gray
    yminorgrid=true, minorgridalpha=0.3, minorgridcolor=:gray,  # Enable minor grid lines
    xminorgrid=false  # Disable minor grid lines for the x-axis
)

# Plot the first point in the subplot
for jj in 1:length(name_network)
    scatter!(
        cc_sub, [1], [mean_cumulative_error[1, jj]],
        yerr=[std_cumulative_error[1, jj]],  # Add error bars
        label=name_network[jj],
        marker=markers[mod1(jj, length(markers))],  # Cycle through marker list
        markersize=6, markerstrokewidth=1, markerstrokecolor=:black,
        alpha=0.9  # Slight transparency for overlapping points
    )
end

# Plot the remaining points in the main plot
for jj in 1:length(name_network)
    scatter!(
        cc_main, 2:length(PNorder_), mean_cumulative_error[2:end, jj],
        yerr=std_cumulative_error[2:end, jj],  # Add error bars
        label=name_network[jj] * " forecast",
        marker=markers[mod1(jj, length(markers))],  # Cycle through marker list
        markersize=6, markerstrokewidth=1, markerstrokecolor=:black,
        alpha=0.9  # Slight transparency for overlapping points
    )
    
    # Plot individual errors as small transparent orange horizontal stripes
    for ii in 1:1
        vecc = PN_errors[ii, jj, 1:NUMBER_OF_EVENTS_TO_BE_USED_SINGLE_CATALOG]
        scatter!(
            cc_sub, fill(ii, length(vecc)), vecc,
            marker=:hline, markersize=15, markerstrokewidth=2, alpha=0.5, color=:orange, label=""
        )
    end

    for ii in 2:length(PNorder_)
        vecc = PN_errors[ii, jj, 1:NUMBER_OF_EVENTS_TO_BE_USED_SINGLE_CATALOG]
        scatter!(
            cc_main, fill(ii, length(vecc)), vecc,
            marker=:hline, markersize=15, markerstrokewidth=2, alpha=0.5, color=:orange, label=""
        )
    end
end

# Add a dummy plot for the legend entry in the main plot
scatter!(
    cc_main, [NaN], [NaN],
    marker=:hline, markersize=15, markerstrokewidth=2, alpha=0.5, color=:orange, label="Errors from single events"
)

# Plot the GWTC-3 results in the main plot
scatter!(
    cc_main, 2:length(PNorder_), LVK_GWTC3_results[2:end],
    label="LVK GWTC-3", marker=:diamond, markersize=6, markerstrokewidth=1, markerstrokecolor=:black, color=:darkblue
)

# Plot the first GWTC-3 result in the subplot
scatter!(
    cc_sub, [1], [LVK_GWTC3_results[1]],
    label="LVK GWTC-3", marker=:diamond, markersize=6, markerstrokewidth=1, markerstrokecolor=:black, color=:darkblue
)

# Combine the main plot and the subplot
plot(cc_sub, cc_main, layout = @layout [a{0.15w} b{0.85w}])

# Save the combined plot to a file on the remote server
savefig(plot_path_prefix * "cumulative_PN_error_plot_PhenomHM_single_errors_LVK_GWTC3_overlaid.png")

# Display the final plot
display(plot(cc_sub, cc_main, layout = @layout [a{0.15w} b{0.85w}]))